# Test of GPT Decoder on exaggeration classification task

In [2]:
%pip install OpenAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 39.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [OpenAI]2m7/8 [OpenAI]c]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

In [2]:
import os
import json
import pandas as pd
from openai import OpenAI
from sklearn.metrics import f1_score, classification_report


In [ ]:

#connect to OpenAI api
client = OpenAI(api_key="sk-proj-XXXXX")

#set labels
label_to_id = {"downplays": 0, "same": 1, "exaggerates": 2}

#define function for predicting class with latest form of chat GPT
def predict_label(abstract_text, press_text, model="gpt-5.4"):
    prompt = f"""
Task:
Compare the press release conclusion to the scientific abstract conclusion.

Labels:
- downplays: the press release makes the claim weaker or more cautious than the abstract
- same: the press release matches the strength of the abstract
- exaggerates: the press release makes the claim stronger, more certain, or more causal than the abstract

Rules:
- Return exactly one label.
- Choose only from: downplays, same, exaggerates.
- Base the decision only on difference in claim strength, not writing quality, topic, or readability.
- Do not explain your answer.

Abstract conclusion:
{abstract_text}

Press release conclusion:
{press_text}
""".strip()

    response = client.responses.create(
        model=model,
        input=prompt,
        #make sure outputs in useful format
        text={
            "format": {
                "type": "json_schema",
                "name": "label_output",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "label": {
                            "type": "string",
                            "enum": ["downplays", "same", "exaggerates"]
                        }
                    },
                    "required": ["label"],
                    "additionalProperties": False
                }
            }
        }
    )

    return json.loads(response.output_text)["label"]

### Test on one fold just to make sure working

In [4]:
from src.data_holdout import get_pooled_df
from src.data_holdout import get_fold_from_disk
from src.data_holdout import get_test_from_disk

/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
full_df = get_pooled_df()

In [13]:

train_df, val_df = get_fold_from_disk(full_df, fold=0)
df = val_df
y_true = df["exaggeration_label"].tolist()
y_pred = [
    label_to_id[predict_label(row["abstract_conclusion"], row["press_release_conclusion"])]
    for _, row in df.iterrows()
]

print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, target_names=["downplays", "same", "exaggerates"]))

Macro F1: 0.3275823311228504
              precision    recall  f1-score   support

   downplays       0.23      0.17      0.19        18
        same       0.60      0.69      0.64        65
 exaggerates       0.17      0.13      0.15        23

    accuracy                           0.48       106
   macro avg       0.33      0.33      0.33       106
weighted avg       0.44      0.48      0.46       106



### Run test data

In [8]:
test_df = get_test_from_disk(full_df)
df = test_df
y_true = df["exaggeration_label"].tolist()
y_pred = [
    label_to_id[predict_label(row["abstract_conclusion"], row["press_release_conclusion"])]
    for _, row in df.iterrows()
]

print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, target_names=["downplays", "same", "exaggerates"]))

Macro F1: 0.3763722491649628
              precision    recall  f1-score   support

   downplays       0.38      0.13      0.19        23
        same       0.66      0.69      0.67        81
 exaggerates       0.23      0.31      0.26        29

    accuracy                           0.51       133
   macro avg       0.42      0.38      0.38       133
weighted avg       0.52      0.51      0.50       133



## Add few shot learning

In [21]:
train_df, val_df = get_fold_from_disk(full_df, fold=0)
few_shot_df = (
    train_df
    .groupby("exaggeration_label", group_keys=False)
    .head(2)
    .reset_index(drop=True)
)
few_shot_df

,original_file_id,press_release_conclusion,press_release_strength,abstract_conclusion,abstract_strength,exaggeration_label
0,11-004,"Having diabetes increases the risk of dying from the effects of a heart attack by around 50 per cent, a University o...",3,"At index acute myocardial infarction, diabetes was common and associated with significant long-term excess mortality...",1,2
1,08-11-021-1,An early study in mice has shown that leukaemic stem cells can be abolished by suppressing two proteins found in the...,2,"Together, these results reveal an important functional interplay between MLL/Hox and Bmi1 in regulating cellular sen...",3,0
2,15-11-015-1,An international study suggests that giving vitamin A supplements to children in low and middle income countries cou...,2,Vitamin A supplementation is associated with large reductions in mortality,1,2
3,02-11-020-1,Plain cigarette packaging could help prevent people taking up the habit but would have little effect on those who al...,2,"Among non-smokers and non-daily cigarette smokers, plain packaging appears to increase visual attention towards heal...",2,1
4,25-055,"Communicating the results of DNA tests has little or no impact on behaviour change, such as stopping smoking or incr...",0,Expectations that communicating DNA based risk estimates changes behaviour is not supported by existing evidence.,0,1
5,32-003,Kent researchers have identified how few mutations it can take for Ebolaviruses to adapt to affect previously resist...,2,"Hence, there is evidence that few mutations including crucial mutations in VP24 enable Ebola virus adaptation to new...",3,0


In [17]:

label_to_id = {"downplays": 0, "same": 1, "exaggerates": 2}
id_to_label = {0: "downplays", 1: "same", 2: "exaggerates"}

def predict_label(abstract_text, press_text, few_shot_df=None, model="gpt-5.4"):
    prompt = """
Task:
Compare the press release conclusion to the scientific abstract conclusion.

Labels:
- downplays: the press release makes the claim weaker or more cautious than the abstract
- same: the press release matches the strength of the abstract
- exaggerates: the press release makes the claim stronger, more certain, or more causal than the abstract

Rules:
- Return exactly one label.
- Choose only from: downplays, same, exaggerates.
- Base the decision only on difference in claim strength, not writing quality, topic, or readability.
- Do not explain your answer.
""".strip()

    if few_shot_df is not None:
        for _, row in few_shot_df.iterrows():
            prompt += f"""

Example:
Abstract conclusion:
{row['abstract_conclusion']}

Press release conclusion:
{row['press_release_conclusion']}

Correct label:
{id_to_label[row['exaggeration_label']]}
"""

    prompt += f"""

Now classify this example.

Abstract conclusion:
{abstract_text}

Press release conclusion:
{press_text}
"""

    response = client.responses.create(
        model=model,
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": "label_output",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "label": {
                            "type": "string",
                            "enum": ["downplays", "same", "exaggerates"]
                        }
                    },
                    "required": ["label"],
                    "additionalProperties": False
                }
            }
        }
    )

    return json.loads(response.output_text)["label"]

### Few shot on val data

In [22]:
df = val_df

y_true = df["exaggeration_label"].tolist()
y_pred = [
    label_to_id[predict_label(
        row["abstract_conclusion"],
        row["press_release_conclusion"],
        few_shot_df=few_shot_df
    )]
    for _, row in df.iterrows()
]

print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, target_names=["downplays", "same", "exaggerates"]))

Macro F1: 0.42329931972789114
              precision    recall  f1-score   support

   downplays       0.41      0.39      0.40        18
        same       0.63      0.62      0.62        65
 exaggerates       0.23      0.26      0.24        23

    accuracy                           0.50       106
   macro avg       0.43      0.42      0.42       106
weighted avg       0.51      0.50      0.50       106



### Few shot on test data

In [23]:
test_df = get_test_from_disk(full_df)
df = test_df

y_true = df["exaggeration_label"].tolist()
y_pred = [
    label_to_id[predict_label(
        row["abstract_conclusion"],
        row["press_release_conclusion"],
        few_shot_df=few_shot_df
    )]
    for _, row in df.iterrows()
]

print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, target_names=["downplays", "same", "exaggerates"]))

Macro F1: 0.40099505130179974
              precision    recall  f1-score   support

   downplays       0.40      0.17      0.24        23
        same       0.67      0.68      0.67        81
 exaggerates       0.24      0.34      0.29        29

    accuracy                           0.52       133
   macro avg       0.44      0.40      0.40       133
weighted avg       0.53      0.52      0.52       133

